# Image Processing Pipeline for Psychophysics

A modular pipeline for loading, transforming, analysing, and exporting face images for use in psychophysics experiments. Each processing step can be independently toggled, configured, and combined to suit your experimental requirements.

*For more details, see: Pang, D. K. F., Hibble, A., Smithson, H. E., & Azzopardi, P. (in preparation). AI-Generated Psychophysics Face Stimuli: Methods, Challenges, and Best Practices.*

## Pipeline Overview

Each step can be independently toggled on or off via the configuration cell (Section 2).

| Step | Description |
|------|-------------|
| 1 | Load images (single file or entire folder) |
| 2 | Convert to greyscale |
| 3 | Face detection & landmark-based alignment and scaling |
| 4 | Scale and crop to target dimensions |
| 5 | Adjust luminance, contrast, and spatial frequency content |
| 6 | Apply oval mask with gradient transition |
| 7 | Compute image analytics (histogram, luminance, contrast, Fourier) |
| 8 | Export processed images |

> **Note on step order:** Adjustments (Step 5) are applied *before* the oval mask (Step 6). This ensures the background colour outside the oval is always exactly the specified value (not modified when adjusting luminance and contrast).

## How to Use

1. Run all cells in **Sections 1–4** to load imports, the face detector, and all processing functions.$^{*}$
2. In **Section 2**, configure the pipeline:
   <br>a. Set `INPUT_PATH` and `OUTPUT_FOLDER` to your image locations.
   <br>b. Toggle `ENABLE_*` flags to turn steps on or off.
   <br>c. Adjust step-specific parameters (alignment, masking, contrast, etc.) as needed.
3. *(Optional)* Run **Section 5**: if `USE_REFERENCE_IMAGE = True` is selected in **Section 2**, target luminance and contrast values are derived from a reference image. If `USE_REFERENCE_IMAGE = False` this section only prints a memo noting that no reference image was used.
4. Run **Section 6** to process all images. Progress and analytics are printed per image.
5. Run **Sections 7–8** for a side-by-side visual comparison and a summary analytics table.

> **Note:** This pipeline will download a file called *"lbfmodel.yaml"* the first tim it is run to locally process facial landmarks (needed to scale and centre the image based on facial features). The file is saved and reused subsequentely.

$^{*}$ If **Section 1** returns an import error, install the required packages by running `pip install opencv-contrib-python matplotlib numpy Pillow scipy` in your terminal.

**Version:** 1.0.0  
**Authors:** Damian K. F. Pang, Alexandra Hibble, Hannah E. Smithson, & Paul Azzopardi <br>University of Oxford<br>
**Last updated:** June 2026  
**License:** MIT

## 1. Imports & Dependencies

In [ ]:
# Uncomment to install missing packages:
# !pip install numpy pillow matplotlib scipy opencv-contrib-python

import os
import urllib.request
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from scipy import fft as scipy_fft

warnings.filterwarnings('ignore')
print('✅ All imports successful.')

## 2. Configuration

All pipeline settings are in the cell below. Toggle each processing step with its `ENABLE_*` flag; set to `True` to run or `False` to skip. No other cells need to be edited during normal use.

In [ ]:
# ═══════════════════════════════════════════════════════
# INPUT / OUTPUT
# ═══════════════════════════════════════════════════════

# Path to a single image file OR a folder containing images.
#INPUT_PATH = "./input"
INPUT_PATH = "../API_Dataset"#"./FaceImages/Raw"
SUPPORTED_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"]

OUTPUT_FOLDER = "./FaceImages/Processed"   # Folder where processed images will be saved.
OUTPUT_FORMAT = "GIF"        # "PNG", "JPEG", "TIFF", "BMP", "WEBP", "GIF"
JPEG_QUALITY  = 95           # Only used when OUTPUT_FORMAT = "JPEG".

# ═══════════════════════════════════════════════════════
# STEP TOGGLES  —  True = enabled,  False = skip
# ═══════════════════════════════════════════════════════
ENABLE_GREYSCALE   = True
ENABLE_FACE_ALIGN  = True    # Face detection, landmark alignment & scaling.
ENABLE_SCALING     = True    # Resize to fit within MAX_WIDTH / MAX_HEIGHT.
ENABLE_CROPPING    = True    # Crop to DESIRED_W × DESIRED_H.
ENABLE_ADJUSTMENTS = True    # Luminance, contrast, and frequency adjustments.
ENABLE_OVAL_MASK   = True    # Apply oval mask over the image.
ENABLE_ANALYTICS   = True    # Compute and display analytics plots.
ENABLE_EXPORT      = True    # Save processed images to OUTPUT_FOLDER.

# ═══════════════════════════════════════════════════════
# FACE ALIGNMENT
# ═══════════════════════════════════════════════════════

# --- Centring ---
# The image will be translated so the chosen reference point lands at the
# image centre (or offset from it by FACE_OFFSET_X / FACE_OFFSET_Y).
#
# Options for FACE_CENTER_MODE:
#   "eye_midpoint"     – midpoint between the two inner eye corners (default)
#   "eye_nose_midpoint"– midpoint between the eye-line and the nose tip
#   "nose_tip"         – nose tip landmark
#   "face_center"      – geometric centre of the detected face bounding box
FACE_CENTER_MODE = "face_center"

# Optional pixel offset from the image centre after alignment.
# Positive = right / down;  negative = left / up.
FACE_OFFSET_X = 0   # pixels
FACE_OFFSET_Y = 0   # pixels

# --- Scaling ---
# The image will be scaled so that the chosen reference measure matches
# the target value in pixels.
#
# Options for FACE_SCALE_MODE:
#   "eye_width"        – distance between the outer eye corners (default)
#   "interpupillary"   – distance between the pupil centres (landmarks 37 & 46)
#   "eye_to_nose"      – distance from eye midpoint to nose tip
#   "face_width"       – width of the detected face bounding box
FACE_SCALE_MODE   = "eye_to_nose"
FACE_SCALE_TARGET = 160   # pixels — the desired size of the chosen measure

# --- Visualisation ---
# If True, facial landmarks are drawn as dots on the analytics preview image.
SHOW_LANDMARKS = False
LANDMARK_COLOR = (0, 255, 0)   # BGR colour for landmark dots
LANDMARK_RADIUS = 2              # dot radius in pixels

# ═══════════════════════════════════════════════════════
# SCALING & CROPPING
# ═══════════════════════════════════════════════════════

# Resize: the image is scaled down to fit within these bounds while
# preserving aspect ratio.  Does not upscale.
MAX_WIDTH  = 400   # pixels
MAX_HEIGHT = 400   # pixels

# Crop: the image is cropped to exactly DESIRED_W × DESIRED_H.
# Set OFFSET_X / OFFSET_Y to None to crop from the centre,
# or provide integer pixel values for the top-left corner.
DESIRED_W = 280    # pixels
DESIRED_H = 380    # pixels
OFFSET_X  = None   # None = centre crop
OFFSET_Y  = None   # None = centre crop

# ═══════════════════════════════════════════════════════
# OVAL MASK
# ═══════════════════════════════════════════════════════

# Size: fractions ≤ 1.0 are interpreted as proportions of the image
# dimension (e.g. 0.8 = 80% of width).  Larger values are pixels.
OVAL_WIDTH  = 0.55   # fraction or pixels
OVAL_HEIGHT = 0.55   # fraction or pixels

# Offset of the oval centre from the image centre in pixels.
# Positive = right / down;  negative = left / up.
OVAL_OFFSET_X = 0   # pixels
OVAL_OFFSET_Y = 0   # pixels

# Background colour outside the oval (R, G, B), values 0–255.
# This colour is applied *exactly* at the outer edge — the gradient
# blends from the image content to this colour over OVAL_GRADIENT_WIDTH pixels.
OVAL_BG_COLOR = (128, 128, 128)   # mid-grey

# Width of the soft transition zone in pixels.
# 0 = hard binary edge.
OVAL_GRADIENT_WIDTH = 60   # pixels

# Shape of the fade curve across the transition zone:
#   "linear" – even, straight ramp
#   "soft"   – slow start, fast finish; feathered, diffuse edge
#   "hard"   – fast start, slow finish; crisper edge
#   "smooth" – S-curve, slow at both ends; most perceptually natural
OVAL_GRADIENT_CURVE = "smooth"

# Override the fade curve with a specific exponent (ignored for "smooth").
#   power < 1  (e.g. 0.5) → hard start, soft finish
#   power = 1             → identical to "linear"
#   power > 1  (e.g. 2.0) → soft start, hard finish
#   power ≥ 3             → very gradual, heavily feathered
# Set to None to use OVAL_GRADIENT_CURVE instead.
OVAL_GRADIENT_POWER = None

# ═══════════════════════════════════════════════════════
# LUMINANCE & CONTRAST ADJUSTMENTS
# ═══════════════════════════════════════════════════════

# Set USE_REFERENCE_IMAGE = True to derive all targets automatically
# from a reference image.  Overrides the manual values below.
USE_REFERENCE_IMAGE  = False
REFERENCE_IMAGE_PATH = "./reference.jpg"

# Target mean luminance (0–255).  Set to None to skip.
# Applied first so peak/floor targets are relative to the new mean.
TARGET_MEAN_LUMINANCE  = 128   # e.g. 128

# Target peak (bright end) and floor (dark end) luminance (0–255).
# Corrections taper progressively to zero at the mean — mid-tones are unaffected.
TARGET_PEAK_LUMINANCE  = None   # e.g. 230
TARGET_FLOOR_LUMINANCE = None   # e.g. 20

# Target RMS contrast (0–127).  Set to None to skip.
# Scales pixel deviations from the mean; mean luminance is preserved.
TARGET_RMS_CONTRAST = 30   # e.g. 40

# ═══════════════════════════════════════════════════════
# FREQUENCY FILTERING
# ═══════════════════════════════════════════════════════
ENABLE_FREQ_FILTER = False

# "low"  — low-pass:  keeps low frequencies, removes high frequencies.
#           Effect: image looks blurred / smoothed.
# "high" — high-pass: keeps high frequencies, removes low frequencies.
#           Effect: image looks like an edge map.
FREQ_FILTER_TYPE = "low"

# Cutoff frequency — gain = 0.5 at exactly this value.
# When OBSERVER_BASED = True:  specify in cpd.
# When OBSERVER_BASED = False: specify as a normalised fraction (0.0–1.0).
FREQ_FILTER_CUTOFF = 0.8   # cpd or normalised

# Transition band width around the cutoff.
# Wider = gentler transition = fewer ringing artefacts.
# Recommended: at least 10–20% of the cutoff value.
FREQ_FILTER_ROLLOFF = 0.16   # cpd or normalised (same units as FREQ_FILTER_CUTOFF)

# ═══════════════════════════════════════════════════════
# ANALYTICS & DISPLAY
# ═══════════════════════════════════════════════════════

# Which frequency plot to include in the analytics dashboard:
#   "radial"    – Radial Power Spectrum (mean amplitude vs. spatial frequency).
#                 Preferred for psychophysics: shows how much energy is present
#                 at each spatial scale, averaged across all orientations.
#   "amplitude" – Amplitude distribution (how many FFT components exist at
#                 each amplitude level).  Diagnostic, not frequency vs. amplitude.
#   "both"      – Show both plots side by side.
ANALYTICS_FREQ_PLOT = "radial"

# Observer-based frequency scaling:
#   True  — frequency axes are in cycles per degree (cpd), the standard unit
#            for psychophysics.  Requires IMAGE_WIDTH_CM and VIEWING_DISTANCE_CM.
#            Plots include perceptual band markers and a CSF reference curve.
#   False — frequency axes are normalised (0 = DC, 1 = Nyquist), independent
#            of display size or viewing distance.
OBSERVER_BASED = True

# Physical display and viewing geometry — only used when OBSERVER_BASED = True.
IMAGE_WIDTH_CM      = 20.0   # width of the image on screen in cm
VIEWING_DISTANCE_CM = 57.0   # observer viewing distance in cm
                             # (at 57 cm, 1 cm of screen = exactly 1° visual angle)

print('✅ Configuration loaded.')

## 3. Face Detection Model Setup

This cell initialises the OpenCV face detector and the LBF facial landmark model.
The landmark model file is downloaded automatically if it is not present locally.

**Run this cell even if `ENABLE_FACE_ALIGN = False`** — the pipeline checks the flag
at runtime, but the model objects must exist in memory.

In [ ]:
# ── Haar cascade face detector ────────────────────────────────────────────
# This is a classical (non-deep-learning) face detector bundled with OpenCV.
# It is fast and reliable for frontal faces under reasonable lighting.
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# ── LBF landmark model ────────────────────────────────────────────────────
# The Local Binary Features (LBF) model detects 68 facial landmarks.
# The model file is not bundled with OpenCV and must be downloaded separately.
# It is cached locally after the first download.
MODEL_FILE = "lbfmodel.yaml"
MODEL_URL  = "https://raw.githubusercontent.com/kurnianggoro/GSOC2017/master/data/lbfmodel.yaml"

if not os.path.exists(MODEL_FILE):
    print(f"Downloading {MODEL_FILE} …")
    urllib.request.urlretrieve(MODEL_URL, MODEL_FILE)
    print("Download complete.")
else:
    print(f"✅ Model file '{MODEL_FILE}' found.")

# ── Initialise landmark detector ──────────────────────────────────────────
try:
    facemark = cv2.face.createFacemarkLBF()
    facemark.loadModel(MODEL_FILE)
    print("✅ Face landmark detector ready.")
except AttributeError:
    print("ERROR: 'cv2.face' module missing. "
          "Install opencv-contrib-python: pip install opencv-contrib-python")
except Exception as e:
    print(f"ERROR initialising landmark detector: {e}")

## 4. Core Functions

### 4.1 Image Loading

In [ ]:
def load_images(
    input_path: str, extensions: List[str]
) -> List[Tuple[str, np.ndarray]]:
    """
    Load images from a single file or a directory.

    Returns a list of (filename, ndarray) tuples.
    Arrays are uint8, shape (H, W, 3) RGB.
    """
    p = Path(input_path)
    paths: List[Path] = []

    if p.is_file():
        paths = [p]
    elif p.is_dir():
        for ext in extensions:
            paths.extend(p.glob(f"*{ext}"))
            paths.extend(p.glob(f"*{ext.upper()}"))
        paths = sorted(set(paths))
    else:
        raise FileNotFoundError(f"Input path not found: {input_path}")

    if not paths:
        raise ValueError(f"No supported images found in: {input_path}")

    images = []
    for path in paths:
        img = Image.open(path).convert("RGB")
        arr = np.array(img, dtype=np.uint8)
        images.append((path.name, arr))
        print(f"  Loaded: {path.name}  {arr.shape[1]}×{arr.shape[0]} px")

    print(f"\n✅ {len(images)} image(s) loaded.")
    return images


print('✅ load_images() defined.')

### 4.2 Greyscale Conversion

In [ ]:
def to_greyscale(img: np.ndarray) -> np.ndarray:
    """
    Convert an RGB image to greyscale using ITU-R BT.709 luminance weights.

    Returns a uint8 array of shape (H, W, 3) — kept as 3-channel so all
    downstream steps remain channel-agnostic.
    """
    weights = np.array([0.2126, 0.7152, 0.0722], dtype=np.float32)
    grey = (img.astype(np.float32) @ weights).astype(np.uint8)   # (H, W)
    return np.stack([grey, grey, grey], axis=-1)                  # (H, W, 3)


print('✅ to_greyscale() defined.')

### 4.3 Face Alignment

Uses OpenCV's Haar cascade detector to locate the face bounding box and the LBF model to identify 68 facial landmarks. The image is then:

1. **Scaled** so the chosen reference measure (e.g. eye-to-nose distance) matches the target pixel value.
2. **Translated** so the chosen centring reference point lands at the image centre.

Landmark coordinates are tracked through both transforms so they remain accurate for any downstream visualisation.

In [ ]:
# ── Landmark index reference (68-point LBF model) ─────────────────────────
# Jaw:          0–16
# Left brow:    17–21   Right brow:  22–26
# Nose bridge:  27–30   Nose base:   31–35
# Left eye:     36–41   Right eye:   42–47
#   36 = outer corner, 39 = inner corner (left eye)
#   42 = inner corner, 45 = outer corner (right eye)
# Outer lips:   48–59   Inner lips:  60–67

def detect_landmarks(
    img: np.ndarray,
) -> Optional[np.ndarray]:
    """
    Detect facial landmarks in *img* (RGB uint8).

    Returns an (68, 2) float32 array of (x, y) landmark coordinates,
    or None if no face is detected.
    """
    grey_cv = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Detect face bounding boxes
    faces = face_cascade.detectMultiScale(
        grey_cv, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )

    if len(faces) == 0:
        return None

    # Use the largest detected face if multiple are found
    faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
    faces_input = np.array([faces[0]])

    ok, landmarks = facemark.fit(grey_cv, faces_input)
    if not ok or len(landmarks) == 0:
        return None

    return landmarks[0][0].astype(np.float32)   # (68, 2)


def _landmark_center(
    lm: np.ndarray,
    mode: str,
    face_box: Optional[Tuple] = None,
) -> Tuple[float, float]:
    """
    Compute the (x, y) reference point used for centring.

    Parameters
    ----------
    lm        : (68, 2) landmark array
    mode      : one of 'eye_midpoint', 'eye_nose_midpoint', 'nose_tip',
                'face_center'
    face_box  : (x, y, w, h) from face detector, required for 'face_center'
    """
    # Pupil centres: average of the 6 points forming each eye
    left_pupil  = lm[36:42].mean(axis=0)   # left eye landmarks 36–41
    right_pupil = lm[42:48].mean(axis=0)   # right eye landmarks 42–47
    eye_midpoint = (left_pupil + right_pupil) / 2
    nose_tip     = lm[30]                  # landmark 30 = nose tip

    if mode == "eye_midpoint":
        return float(eye_midpoint[0]), float(eye_midpoint[1])
    elif mode == "eye_nose_midpoint":
        pt = (eye_midpoint + nose_tip) / 2
        return float(pt[0]), float(pt[1])
    elif mode == "nose_tip":
        return float(nose_tip[0]), float(nose_tip[1])
    elif mode == "face_center":
        if face_box is None:
            raise ValueError("face_box required for mode='face_center'")
        x, y, w, h = face_box
        return float(x + w / 2), float(y + h / 2)
    else:
        raise ValueError(f"Unknown FACE_CENTER_MODE: '{mode}'")


def _landmark_scale_measure(
    lm: np.ndarray,
    mode: str,
    face_box: Optional[Tuple] = None,
) -> float:
    """
    Compute the reference distance used for scaling.

    Parameters
    ----------
    lm        : (68, 2) landmark array
    mode      : one of 'eye_width', 'interpupillary', 'eye_to_nose',
                'face_width'
    face_box  : (x, y, w, h) from face detector, required for 'face_width'
    """
    left_outer  = lm[36]   # outer corner of left eye
    right_outer = lm[45]   # outer corner of right eye
    left_pupil  = lm[36:42].mean(axis=0)
    right_pupil = lm[42:48].mean(axis=0)
    eye_midpoint = (left_pupil + right_pupil) / 2
    nose_tip     = lm[30]

    if mode == "eye_width":
        return float(np.linalg.norm(right_outer - left_outer))
    elif mode == "interpupillary":
        return float(np.linalg.norm(right_pupil - left_pupil))
    elif mode == "eye_to_nose":
        return float(np.linalg.norm(nose_tip - eye_midpoint))
    elif mode == "face_width":
        if face_box is None:
            raise ValueError("face_box required for mode='face_width'")
        return float(face_box[2])
    else:
        raise ValueError(f"Unknown FACE_SCALE_MODE: '{mode}'")


def align_face(
    img: np.ndarray,
    center_mode: str      = "eye_midpoint",
    scale_mode: str       = "eye_width",
    scale_target: int     = 80,
    offset_x: int         = 0,
    offset_y: int         = 0,
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Align *img* by scaling and translating based on facial landmarks.

    1. Detects landmarks and face bounding box.
    2. Scales the image so that *scale_mode* measure = *scale_target* pixels.
    3. Translates so that the *center_mode* reference point lands at the
       image centre plus (*offset_x*, *offset_y*).

    Returns (aligned_image, landmarks_in_aligned_image).
    If no face is detected, returns the original image unchanged and None.
    """
    lm = detect_landmarks(img)
    if lm is None:
        print("    ⚠️  No face detected — skipping alignment.")
        return img, None

    # Detect face bounding box — only needed for 'face_center' / 'face_width' modes
    face_box = None
    if center_mode == 'face_center' or scale_mode == 'face_width':
        grey_cv  = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        faces    = face_cascade.detectMultiScale(grey_cv, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        face_box = tuple(int(v) for v in sorted(faces, key=lambda f: f[2] * f[3], reverse=True)[0]) if len(faces) > 0 else None

    H, W = img.shape[:2]

    # ── Step 1: Scale ──────────────────────────────────────────────────────
    current_measure = _landmark_scale_measure(lm, scale_mode, face_box)
    if current_measure < 1.0:
        print("    ⚠️  Scale measure too small — skipping scaling.")
        scale_factor = 1.0
    else:
        scale_factor = scale_target / current_measure

    new_w = max(1, int(round(W * scale_factor)))
    new_h = max(1, int(round(H * scale_factor)))
    img_scaled = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
    lm_scaled  = lm * scale_factor   # scale landmark coordinates too

    # Scale the face bounding box to match the resized image
    face_box_scaled = (
        tuple(v * scale_factor for v in face_box) if face_box is not None else None
    )

    # ── Step 2: Translate ──────────────────────────────────────────────────
    ref_x, ref_y = _landmark_center(lm_scaled, center_mode, face_box_scaled)
    target_x = new_w / 2 + offset_x
    target_y = new_h / 2 + offset_y
    dx = target_x - ref_x
    dy = target_y - ref_y

    M = np.float32([[1, 0, dx], [0, 1, dy]])   # translation matrix
    img_aligned = cv2.warpAffine(
        img_scaled, M, (new_w, new_h),
        flags=cv2.INTER_LANCZOS4,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=OVAL_BG_COLOR[::-1],   # fill new border with bg colour
    )

    # Translate landmark coordinates to match
    lm_aligned = lm_scaled.copy()
    lm_aligned[:, 0] += dx
    lm_aligned[:, 1] += dy

    return img_aligned, lm_aligned


def draw_landmarks(
    img: np.ndarray,
    landmarks: np.ndarray,
    color: Tuple[int, int, int] = (0, 200, 255),
    radius: int = 2,
) -> np.ndarray:
    """
    Draw landmark dots on a copy of *img* for visualisation.
    Uses BGR colour order (OpenCV convention).
    Returns a uint8 RGB array.
    """
    # Convert RGB → BGR for OpenCV drawing, then back
    vis = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    for (x, y) in landmarks.astype(int):
        cv2.circle(vis, (x, y), radius, color, -1, lineType=cv2.LINE_AA)
    return cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)


print('✅ Face alignment functions defined.')

### 4.4 Scaling & Cropping

In [ ]:
def scale_image(img: np.ndarray, max_w: int, max_h: int) -> np.ndarray:
    """
    Resize *img* to fit within *max_w* × *max_h* while preserving aspect ratio.
    Only downscales — images already smaller than the bounds are returned unchanged.
    """
    h, w = img.shape[:2]
    if w <= max_w and h <= max_h:
        return img
    scale = min(max_w / w, max_h / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)


def crop_image(
    img: np.ndarray,
    crop_w: int, crop_h: int,
    offset_x: Optional[int] = None,
    offset_y: Optional[int] = None,
) -> np.ndarray:
    """
    Crop *img* to *crop_w* × *crop_h*.

    *offset_x* / *offset_y* define the top-left corner of the crop window.
    Set to None to crop from the image centre.
    """
    h, w = img.shape[:2]
    crop_w = min(crop_w, w)
    crop_h = min(crop_h, h)

    start_x = (w - crop_w) // 2 if offset_x is None else max(0, min(offset_x, w - crop_w))
    start_y = (h - crop_h) // 2 if offset_y is None else max(0, min(offset_y, h - crop_h))

    return img[start_y:start_y + crop_h, start_x:start_x + crop_w]


print('✅ scale_image() / crop_image() defined.')

### 4.5 Oval Mask

The mask is a float32 array where `1.0` = fully inside (keep image) and `0.0` = fully
outside (show background).  The transition zone blends smoothly between these values.

**Why adjustments must come before the mask:**  
The compositing formula is `result = mask × image + (1 − mask) × bg_color`.  
Since `(1 − mask) × bg_color` is exactly `bg_color` wherever `mask = 0`, the background
is always a pure, exact colour — provided the image has already been adjusted *before*
the mask is applied.  Applying adjustments after the mask would alter the background pixels too.

In [ ]:
def _resolve_dim(value: float, image_dim: int) -> int:
    """Return pixel size from a fraction (≤ 1.0) or an absolute pixel value."""
    if 0.0 < value <= 1.0:
        return int(round(value * image_dim))
    return int(round(value))


def build_oval_mask(
    height: int, width: int,
    oval_w: float, oval_h: float,
    offset_x: int = 0, offset_y: int = 0,
    gradient_width: int = 20,
    gradient_curve: str = "linear",
    power: Optional[float] = None,
) -> np.ndarray:
    """
    Create a soft oval mask of shape (H, W), float32 in [0, 1].

    Parameters
    ----------
    oval_w / oval_h  : Oval dimensions.  Fractions ≤ 1.0 are proportions of
                       the image dimension; larger values are pixels.
    offset_x/y       : Centre offset from image centre in pixels.
    gradient_width   : Transition zone width in pixels (0 = hard edge).
    gradient_curve   : 'linear', 'soft', 'hard', or 'smooth'.
    power            : Override exponent (ignored when curve = 'smooth').
    """
    pw = _resolve_dim(oval_w, width)
    ph = _resolve_dim(oval_h, height)
    cx = width  // 2 + offset_x
    cy = height // 2 + offset_y

    yy, xx = np.meshgrid(
        np.arange(height, dtype=np.float64),
        np.arange(width,  dtype=np.float64),
        indexing='ij',
    )

    # Normalised elliptical distance: 1.0 = on the boundary, >1 = outside
    dist = np.sqrt(((xx - cx) / (pw / 2)) ** 2 +
                   ((yy - cy) / (ph / 2)) ** 2)

    if gradient_width <= 0:
        return (dist <= 1.0).astype(np.float32)

    g     = gradient_width / (min(pw, ph) / 2)
    inner = 1.0 - g / 2   # dist at which mask = 1.0 (fully inside)
    outer = 1.0 + g / 2   # dist at which mask = 0.0 (fully outside)

    # t = 0 inside, t = 1 outside; transition between inner and outer
    t = np.clip((dist - inner) / (outer - inner), 0.0, 1.0).astype(np.float32)

    curve = gradient_curve.lower().strip()

    if curve == "smooth":
        mask = 1.0 - (3 * t**2 - 2 * t**3)   # smoothstep S-curve
    else:
        if power is not None:
            if power <= 0:
                raise ValueError(f"power must be a positive float, got {power}")
            exp = float(power)
        else:
            exp = {"linear": 1.0, "soft": 2.0, "hard": 0.5}.get(curve, 1.0)
            if curve not in ("linear", "soft", "hard"):
                raise ValueError(
                    f"Unknown gradient_curve '{gradient_curve}'. "
                    "Choose from: 'linear', 'soft', 'hard', 'smooth'."
                )
        mask = 1.0 - t ** exp

    return mask.astype(np.float32)


def apply_oval_mask(
    img: np.ndarray,
    oval_w: float, oval_h: float,
    offset_x: int = 0, offset_y: int = 0,
    bg_color: Tuple[int, int, int] = (128, 128, 128),
    gradient_width: int = 20,
    gradient_curve: str = "linear",
    power: Optional[float] = None,
) -> np.ndarray:
    """
    Composite *img* against *bg_color* using a soft oval mask.

    result = mask × img + (1 − mask) × bg_color

    The background colour is exact (not blended) outside the transition zone.
    To preserve the exact background colour, always apply image adjustments
    *before* calling this function.
    """
    H, W = img.shape[:2]
    mask = build_oval_mask(
        H, W, oval_w, oval_h,
        offset_x=offset_x, offset_y=offset_y,
        gradient_width=gradient_width,
        gradient_curve=gradient_curve,
        power=power,
    )
    alpha     = mask[:, :, np.newaxis]              # (H, W, 1)
    bg        = np.array(bg_color, dtype=np.float32)
    composite = alpha * img.astype(np.float32) + (1.0 - alpha) * bg
    return np.clip(composite, 0, 255).astype(np.uint8)


print('✅ build_oval_mask() / apply_oval_mask() defined.')

### 4.6 Luminance & Contrast Adjustments

In [ ]:
def compute_luminance(img: np.ndarray) -> np.ndarray:
    """Per-pixel luminance (float32, shape H×W) from an RGB array."""
    weights = np.array([0.2126, 0.7152, 0.0722], dtype=np.float32)
    return img.astype(np.float32) @ weights   # (H, W)


def adjust_mean_luminance(
    img: np.ndarray,
    target_mean: float,
    mask: Optional[np.ndarray] = None,   # float32 (H, W), values 0–1
) -> np.ndarray:
    """
    Shift pixel values so the mean luminance of the *masked region* reaches
    *target_mean*.  The same shift is applied to all pixels; the mask is used
    only to compute the current mean — not to restrict which pixels are changed.
    The oval composite applied afterwards handles the actual region boundary.
    """
    lum     = compute_luminance(img)
    if mask is not None:
        # Weighted mean: pixels with higher mask values contribute more
        current = float(np.average(lum, weights=mask))
    else:
        current = float(lum.mean())
    delta = target_mean - current
    return np.clip(img.astype(np.float32) + delta, 0, 255).astype(np.uint8)


def adjust_rms_contrast(
    img: np.ndarray,
    target_rms: float,
    mask: Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Scale pixel deviations from the mean to achieve *target_rms* contrast.
    Statistics are computed over the masked region only.
    """
    lum = compute_luminance(img)
    if mask is not None:
        mu = float(np.average(lum, weights=mask))
        current_rms = float(np.sqrt(np.average((lum - mu) ** 2, weights=mask)))
    else:
        mu = float(lum.mean())
        current_rms = float(np.sqrt(np.mean((lum - mu) ** 2)))
    if current_rms < 1e-6:
        return img.copy()
    scale    = target_rms / current_rms
    adjusted = mu + (img.astype(np.float32) - mu) * scale
    return np.clip(adjusted, 0, 255).astype(np.uint8)


def adjust_peak_luminance(
    img: np.ndarray,
    target_peak: float,
    mean_lum: Optional[float] = None,
    mask: Optional[np.ndarray] = None,
) -> np.ndarray:
    lum  = compute_luminance(img)
    if mask is not None:
        mu   = mean_lum if mean_lum is not None else float(np.average(lum, weights=mask))
        # Percentile over masked pixels only (replicate by weighting)
        flat_lum  = lum[mask > 0.5]   # use pixels that are meaningfully inside
        peak = float(np.percentile(flat_lum, 99)) if len(flat_lum) else float(np.percentile(lum, 99))
    else:
        mu   = mean_lum if mean_lum is not None else float(lum.mean())
        peak = float(np.percentile(lum, 99))
    shift = _smooth_transition_curve(lum, mu, peak, target_peak)[:, :, np.newaxis]
    return np.clip(img.astype(np.float32) + shift, 0, 255).astype(np.uint8)


def adjust_floor_luminance(
    img: np.ndarray,
    target_floor: float,
    mean_lum: Optional[float] = None,
    mask: Optional[np.ndarray] = None,
) -> np.ndarray:
    lum   = compute_luminance(img)
    if mask is not None:
        mu    = mean_lum if mean_lum is not None else float(np.average(lum, weights=mask))
        flat_lum = lum[mask > 0.5]
        floor = float(np.percentile(flat_lum, 1)) if len(flat_lum) else float(np.percentile(lum, 1))
    else:
        mu    = mean_lum if mean_lum is not None else float(lum.mean())
        floor = float(np.percentile(lum, 1))
    shift = _smooth_transition_curve(lum, mu, floor, target_floor)[:, :, np.newaxis]
    return np.clip(img.astype(np.float32) + shift, 0, 255).astype(np.uint8)


def _smooth_transition_curve(
    lum: np.ndarray,
    mean_lum: float,
    extreme_lum: float,
    target_extreme: float,
) -> np.ndarray:
    """
    Per-pixel shift that tapers from (extreme_lum → target_extreme) to zero at mean_lum.

    Works for both peak (extreme > mean) and floor (extreme < mean) corrections.
    A quadratic weight ensures mid-tones are barely affected while the extremes
    receive the full correction.
    """
    total_delta = target_extreme - extreme_lum
    span        = extreme_lum - mean_lum
    if abs(span) < 1e-6:
        return np.zeros_like(lum, dtype=np.float32)
    t = np.clip((lum - mean_lum) / span, 0.0, 1.0)
    return (t ** 2 * total_delta).astype(np.float32)

print('✅ Luminance & contrast adjustment functions defined.')

### 4.7 Frequency Filtering

In [ ]:
def compute_pixels_per_degree(
    image_width_px: int,
    image_width_cm: float,
    viewing_distance_cm: float,
) -> float:
    """
    Convert display geometry into pixels per degree of visual angle.

    At 57 cm viewing distance, 1 cm on screen = exactly 1° visual angle,
    which makes the arithmetic particularly convenient.
    """
    degrees_per_cm = np.degrees(np.arctan(1.0 / viewing_distance_cm))
    pixels_per_cm  = image_width_px / image_width_cm
    return pixels_per_cm / degrees_per_cm


def apply_frequency_filter(
    img: np.ndarray,
    filter_type: str = "low",
    cutoff: float    = 0.2,
    rolloff: float   = 0.05,
) -> np.ndarray:
    """
    Apply a low-pass or high-pass Fourier filter to each channel independently.

    Parameters
    ----------
    filter_type : 'low' or 'high'
    cutoff      : Half-power frequency.  In cpd when OBSERVER_BASED = True,
                  normalised (0–1) otherwise.
    rolloff     : Transition band width.  Same units as cutoff.
                  Use at least 10–20% of the cutoff to avoid ringing artefacts.
    """
    H, W = img.shape[:2]

    # ── Convert to normalised units ────────────────────────────────────────
    if OBSERVER_BASED:
        ppd         = compute_pixels_per_degree(W, IMAGE_WIDTH_CM, VIEWING_DISTANCE_CM)
        nyquist_cpd = ppd / 2.0
        if cutoff > nyquist_cpd:
            print(f"    ⚠️  Cutoff ({cutoff} cpd) > Nyquist ({nyquist_cpd:.1f} cpd) — clamped.")
            cutoff = nyquist_cpd
        cutoff_norm  = cutoff  / nyquist_cpd
        rolloff_norm = rolloff / nyquist_cpd
    else:
        cutoff_norm  = float(np.clip(cutoff,  0.0, 1.0))
        rolloff_norm = float(np.clip(rolloff, 0.0, 1.0))

    # ── Normalised radial frequency map ───────────────────────────────────
    fy   = scipy_fft.fftshift(scipy_fft.fftfreq(H))
    fx   = scipy_fft.fftshift(scipy_fft.fftfreq(W))
    FX, FY = np.meshgrid(fx, fy)
    freq = np.clip(np.sqrt(FX**2 + FY**2) * np.sqrt(2), 0, 1)

    # ── Filter mask ────────────────────────────────────────────────────────
    # Smooth sigmoid centred at cutoff_norm.
    # rolloff / 5 maps the full rolloff band to the sigmoid's active range.
    if rolloff_norm > 0:
        low_pass = 1.0 / (1.0 + np.exp((freq - cutoff_norm) / (rolloff_norm / 5)))
    else:
        # Hard brick-wall cut — causes ringing artefacts; use with caution.
        low_pass = (freq <= cutoff_norm).astype(np.float32)

    filt = low_pass if filter_type == "low" else (1.0 - low_pass)

    # ── Apply channel by channel ───────────────────────────────────────────
    result = np.zeros_like(img, dtype=np.float32)
    for c in range(img.shape[2]):
        ch  = img[:, :, c].astype(np.float32) / 255.0
        Fs  = scipy_fft.fftshift(scipy_fft.fft2(ch))
        ch2 = np.real(scipy_fft.ifft2(scipy_fft.ifftshift(Fs * filt)))
        result[:, :, c] = ch2 * 255.0

    return np.clip(result, 0, 255).astype(np.uint8)


print('✅ apply_frequency_filter() / compute_pixels_per_degree() defined.')

### 4.8 Image Analytics

Computes luminance statistics, RMS contrast, and a 2-D Fourier transform for each image, then renders the analytics dashboard. When `SHOW_LANDMARKS = True`, facial landmark positions are overlaid on the image preview.

In [ ]:
def freq_axis_and_labels(
    image_width_px: int,
    n_bins: int,
) -> Tuple[np.ndarray, str, Optional[float]]:
    """
    Return frequency axis values, x-axis label, and Nyquist limit.

    OBSERVER_BASED = True  → axis in cpd
    OBSERVER_BASED = False → axis normalised 0–1
    """
    freq_bins = np.linspace(0, 1, n_bins + 1)
    bin_mids  = (freq_bins[:-1] + freq_bins[1:]) / 2
    if OBSERVER_BASED:
        ppd         = compute_pixels_per_degree(image_width_px, IMAGE_WIDTH_CM, VIEWING_DISTANCE_CM)
        nyquist_cpd = ppd / 2.0
        return bin_mids * nyquist_cpd, "Spatial frequency (cycles per degree, cpd)", nyquist_cpd
    return bin_mids, "Normalised spatial frequency (0 = DC, 1 = Nyquist)", None


def compute_analytics(
    img: np.ndarray, name: str = "", show_plots: bool = True
) -> Dict:
    """
    Compute image analytics and (optionally) display the dashboard.

    Returns a dict with keys:
      histogram, mean_luminance, peak_luminance, floor_luminance,
      rms_contrast, fft_magnitude, fft_phase, fft_freq_map, band_power
    """
    lum = compute_luminance(img)   # (H, W) float32
    H, W = lum.shape

    # ── Brightness histogram ───────────────────────────────────────────────
    hist, bin_edges = np.histogram(lum.ravel(), bins=256, range=(0, 256))

    # ── Luminance statistics ───────────────────────────────────────────────
    mean_lum  = float(lum.mean())
    peak_lum  = float(np.percentile(lum, 99))   # 99th percentile = robust peak
    floor_lum = float(np.percentile(lum,  1))   # 1st  percentile = robust floor

    # ── RMS contrast ──────────────────────────────────────────────────────
    rms_contrast = float(np.sqrt(np.mean((lum - mean_lum) ** 2)))

    # ── 2-D Fourier transform ──────────────────────────────────────────────
    F_shift   = scipy_fft.fftshift(scipy_fft.fft2(lum / 255.0))
    magnitude = np.abs(F_shift)
    phase     = np.angle(F_shift)

    # Normalised radial frequency map (0 = DC, 1 = Nyquist diagonal)
    fy = scipy_fft.fftshift(scipy_fft.fftfreq(H))
    fx = scipy_fft.fftshift(scipy_fft.fftfreq(W))
    FX, FY   = np.meshgrid(fx, fy)
    freq_map = np.clip(np.sqrt(FX**2 + FY**2) * np.sqrt(2), 0, 1)

    # ── Perceptual frequency band power (cpd) ─────────────────────────────
    band_power = {}
    if OBSERVER_BASED:
        ppd          = compute_pixels_per_degree(W, IMAGE_WIDTH_CM, VIEWING_DISTANCE_CM)
        nyquist_cpd  = ppd / 2.0
        freq_map_cpd = freq_map * nyquist_cpd
        total_power  = float(magnitude.sum())
        for key, lo, hi in [
            ("below_1cpd",  0,   1),
            ("1_to_6cpd",   1,   6),
            ("6_to_12cpd",  6,  12),
            ("above_12cpd", 12, nyquist_cpd),
        ]:
            ring = (freq_map_cpd >= lo) & (freq_map_cpd < hi)
            band_power[key] = {
                "absolute": float(magnitude[ring].sum()),
                "percent":  float(magnitude[ring].sum() / total_power * 100),
            }

    analytics = dict(
        histogram       = (hist, bin_edges),
        mean_luminance  = mean_lum,
        peak_luminance  = peak_lum,
        floor_luminance = floor_lum,
        rms_contrast    = rms_contrast,
        fft_magnitude   = magnitude,
        fft_phase       = phase,
        fft_freq_map    = freq_map,
        band_power      = band_power,
    )

    # ── Print summary ──────────────────────────────────────────────────────
    title = f" – {name}" if name else ""
    print(f"\n📊 Analytics{title}")
    print(f"   {'─' * 45}")
    print(f"   Luminance")
    print(f"     Mean          : {mean_lum:7.2f}  (0–255)")
    print(f"     Peak          : {peak_lum:7.2f}  (99th percentile)")
    print(f"     Floor         : {floor_lum:7.2f}  (1st percentile)")
    print(f"     Range         : {peak_lum - floor_lum:7.2f}  (peak − floor)")
    print(f"   {'─' * 45}")
    print(f"   Contrast")
    print(f"     RMS contrast  : {rms_contrast:7.2f}")
    print(f"     Normalised RMS: {rms_contrast / 127.5:7.3f}  (0–1 scale)")
    print(f"   {'─' * 45}")
    if band_power:
        ppd         = compute_pixels_per_degree(W, IMAGE_WIDTH_CM, VIEWING_DISTANCE_CM)
        nyquist_cpd = ppd / 2.0
        print(f"   Spatial Frequency  (Nyquist = {nyquist_cpd:.1f} cpd)")
        print(f"     {'Band':<24} {'Absolute':>12}  {'% of total':>10}")
        print(f"     {'─' * 50}")
        for key, lbl in [
            ("below_1cpd",  "< 1 cpd  (coarse)"),
            ("1_to_6cpd",   "1–6 cpd  (peak CSF)"),
            ("6_to_12cpd",  "6–12 cpd (fine detail)"),
            ("above_12cpd", "> 12 cpd (sub-threshold)"),
        ]:
            bp = band_power[key]
            print(f"     {lbl:<24} {bp['absolute']:>12.1f}  {bp['percent']:>9.1f}%")
    else:
        print("   Spatial frequency bands: set OBSERVER_BASED = True to enable.")
    print(f"   {'─' * 45}")

    if show_plots:
        _plot_analytics(img, analytics, name)

    return analytics


def _csf_curve(freqs: np.ndarray) -> np.ndarray:
    """
    Approximate contrast sensitivity function (Mannos & Sakrison 1974).
    Returns relative sensitivity (peak = 1.0) at each cpd value.
    """
    f   = np.clip(freqs, 0.1, None)
    csf = 2.6 * (0.0192 + 0.114 * f) * np.exp(-(0.114 * f) ** 1.1)
    csf = np.clip(csf, 0, None)
    return csf / csf.max()


def _plot_radial_spectrum(
    ax_main, ax_zoom, analytics: Dict, text_color: str, accent1: str
) -> None:
    """Plot Radial Power Spectrum on two axes: full range and DC-removed zoom."""
    freq_map       = analytics["fft_freq_map"]
    magnitude      = analytics["fft_magnitude"]
    image_width_px = magnitude.shape[1]

    cpd_bands = [
        (0,    1,    "#334455", "< 1 cpd\ncoarse"),
        (1,    6,    "#1a3a2a", "1–6 cpd\npeak CSF"),
        (6,    12,   "#2a2a1a", "6–12 cpd\nfine detail"),
        (12,   None, "#2a1a1a", "> 12 cpd\nfades out"),
    ]

    N_BINS    = 200
    freq_bins = np.linspace(0, 1, N_BINS + 1)
    power_spectrum = np.zeros(N_BINS)
    for i in range(N_BINS):
        ring = (freq_map >= freq_bins[i]) & (freq_map < freq_bins[i + 1])
        if ring.any():
            power_spectrum[i] = magnitude[ring].mean()

    x_vals, xlabel, nyquist_cpd = freq_axis_and_labels(image_width_px, N_BINS)

    def shade_bands(ax, y_max, log_x=False):
        if not OBSERVER_BASED or nyquist_cpd is None:
            return
        for lo, hi, colour, lbl in cpd_bands:
            hi_clipped = min(hi, nyquist_cpd) if hi is not None else nyquist_cpd
            if lo >= nyquist_cpd:
                continue
            ax.axvspan(lo, hi_clipped, color=colour, alpha=0.5, zorder=0)
            lx = np.sqrt(lo * hi_clipped) if (log_x and lo > 0) else (lo + hi_clipped) / 2
            ax.text(lx, y_max * 0.97, lbl, color="#aabbcc", fontsize=6,
                    ha="center", va="top", linespacing=1.3)

    # ── Main plot ──────────────────────────────────────────────────────────
    y_max_main = power_spectrum.max() * 1.05
    shade_bands(ax_main, y_max_main)
    ax_main.fill_between(x_vals, power_spectrum, alpha=0.4, color=accent1, zorder=2)
    ax_main.plot(x_vals, power_spectrum, color=accent1, lw=1.5, zorder=3)
    ax_main.set_xlabel(xlabel, color=text_color, fontsize=8)
    ax_main.set_ylabel("Mean amplitude", color=text_color, fontsize=8)
    ax_main.set_title("Radial Power Spectrum", color=text_color, fontsize=9, pad=5)
    ax_main.set_xlim(x_vals[0], x_vals[-1])

    # ── Zoomed plot (DC removed, log x, y capped) ─────────────────────────
    x_zoom    = x_vals[1:]
    y_zoom    = power_spectrum[1:]
    y_ceiling = np.percentile(y_zoom[y_zoom > 0], 98) * 1.2

    shade_bands(ax_zoom, y_ceiling, log_x=True)
    ax_zoom.fill_between(x_zoom, y_zoom, alpha=0.4, color="#48cae4", zorder=2)
    ax_zoom.plot(x_zoom, y_zoom, color="#48cae4", lw=1.5, zorder=3)

    if OBSERVER_BASED and nyquist_cpd is not None:
        csf_x = np.linspace(x_zoom[0], x_zoom[-1], 300)
        ax_zoom.plot(csf_x, _csf_curve(csf_x) * y_ceiling * 0.85,
                     color="#ff9f1c", lw=1.2, ls="--", alpha=0.8,
                     zorder=4, label="CSF (approx.)")
        ax_zoom.legend(fontsize=7, facecolor="#0d1b2a",
                       edgecolor="#556677", labelcolor=text_color)

    ax_zoom.set_xscale("log")
    ax_zoom.set_ylim(0, y_ceiling)
    ax_zoom.set_xlim(x_zoom[0], x_zoom[-1])
    ax_zoom.set_xlabel(
        "Spatial frequency — cpd (log scale)" if OBSERVER_BASED else "Frequency (log scale)",
        color=text_color, fontsize=8,
    )
    ax_zoom.set_ylabel("Mean amplitude", color=text_color, fontsize=8)
    ax_zoom.set_title(
        "Radial Power Spectrum — Zoomed  ·  DC removed" +
        ("  ·  CSF overlaid" if OBSERVER_BASED else ""),
        color=text_color, fontsize=9, pad=5,
    )


def _plot_freq_histogram(
    ax, analytics: Dict, text_color: str, accent2: str
) -> None:
    """
    Amplitude distribution plot.

    Shows how many FFT components exist at each amplitude level.
    A spike near zero is normal; a long tail indicates a few dominant frequencies.
    Note: this is NOT a frequency vs. amplitude plot — use the Radial Power
    Spectrum for that.
    """
    magnitude  = analytics["fft_magnitude"].ravel()
    log_bins   = np.logspace(0, np.log10(magnitude.max() + 1), 200)
    hist, edges = np.histogram(magnitude, bins=log_bins)
    bin_mids    = (edges[:-1] + edges[1:]) / 2

    ax.fill_between(bin_mids, hist, alpha=0.5, color=accent2)
    ax.plot(bin_mids, hist, color=accent2, lw=1.5)
    ax.set_xscale("log")
    ax.set_xlabel("Amplitude (log scale)", color=text_color, fontsize=8)
    ax.set_ylabel("Number of frequency components", color=text_color, fontsize=8)
    ax.set_title("Amplitude Distribution", color=text_color, fontsize=9, pad=5)


def _plot_analytics(
    img: np.ndarray, analytics: Dict, name: str = "",
    landmarks: Optional[np.ndarray] = None,
) -> None:
    """
    Render the analytics dashboard.

    Grid layout (4 columns):
      Top row    : [image preview] [brightness histogram × 3 cols]
      Bottom row : [FFT magnitude] [radial main] [radial zoom] [amplitude dist?]

    Parameters
    ----------
    landmarks : (68, 2) array or None.  If provided and SHOW_LANDMARKS = True,
                landmark dots are drawn on the preview image.
    """
    hist, bin_edges = analytics["histogram"]
    magnitude = analytics["fft_magnitude"]
    mean_lum  = analytics["mean_luminance"]
    peak_lum  = analytics["peak_luminance"]
    floor_lum = analytics["floor_luminance"]
    rms_c     = analytics["rms_contrast"]

    show_radial   = ANALYTICS_FREQ_PLOT in ("radial", "both")
    show_freqhist = ANALYTICS_FREQ_PLOT in ("amplitude", "both")

    dark       = "#16213e"
    text_color = "#e2e2e2"
    accent1    = "#00b4d8"
    accent2    = "#f77f00"

    fig = plt.figure(figsize=(20, 9), facecolor="#1a1a2e")
    fig.suptitle(
        f"Image Analytics{'  —  ' + name if name else ''}",
        fontsize=15, color="white", fontweight="bold", y=0.98,
    )

    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.42, wspace=0.35,
                           left=0.05, right=0.97, top=0.93, bottom=0.07)

    ax_img  = fig.add_subplot(gs[0, 0])
    ax_hist = fig.add_subplot(gs[0, 1:])
    ax_fft  = fig.add_subplot(gs[1, 0])

    if show_radial and show_freqhist:
        ax_rad_main = fig.add_subplot(gs[1, 1])
        ax_rad_zoom = fig.add_subplot(gs[1, 2])
        ax_freqhist = fig.add_subplot(gs[1, 3])
    elif show_radial:
        ax_rad_main = fig.add_subplot(gs[1, 1:3])
        ax_rad_zoom = fig.add_subplot(gs[1, 3])
        ax_freqhist = None
    else:
        ax_rad_main = None
        ax_rad_zoom = None
        ax_freqhist = fig.add_subplot(gs[1, 1:])

    for ax in [a for a in [ax_img, ax_hist, ax_fft, ax_rad_main, ax_rad_zoom, ax_freqhist]
               if a is not None]:
        ax.set_facecolor(dark)
        ax.tick_params(colors=text_color, labelsize=8)
        for spine in ax.spines.values():
            spine.set_color("#444466")

    # ── Image preview (with optional landmark overlay) ─────────────────────
    preview = img
    if SHOW_LANDMARKS and landmarks is not None:
        preview = draw_landmarks(img, landmarks, LANDMARK_COLOR, LANDMARK_RADIUS)
    ax_img.imshow(preview)
    ax_img.axis("off")
    lm_note = "  (landmarks)" if (SHOW_LANDMARKS and landmarks is not None) else ""
    ax_img.set_title(f"Image Preview{lm_note}", color=text_color, fontsize=9, pad=5)

    # ── Brightness histogram ───────────────────────────────────────────────
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    cmap_bw = LinearSegmentedColormap.from_list("bw", ["black", "white"])
    ax_hist.bar(bin_centers, hist, width=1.0,
                color=cmap_bw(bin_centers / 255.0), alpha=0.9)
    ax_hist.axvline(mean_lum,  color=accent1, lw=1.5, label=f"Mean {mean_lum:.1f}")
    ax_hist.axvline(peak_lum,  color=accent2, lw=1.2, ls="--", label=f"Peak {peak_lum:.1f}")
    ax_hist.axvline(floor_lum, color="#e63946", lw=1.2, ls="--", label=f"Floor {floor_lum:.1f}")
    ax_hist.set_xlim(0, 255)
    ax_hist.set_xlabel("Brightness (0 = black → 255 = white)", color=text_color, fontsize=8)
    ax_hist.set_ylabel("Pixel count", color=text_color, fontsize=8)
    ax_hist.legend(fontsize=8, facecolor=dark, edgecolor="#444466", labelcolor=text_color)
    ax_hist.text(0.98, 0.92, f"RMS contrast: {rms_c:.2f}",
                 transform=ax_hist.transAxes, ha="right", va="top",
                 color=text_color, fontsize=8,
                 bbox=dict(facecolor=dark, edgecolor="#444466", boxstyle="round,pad=0.3"))
    ax_hist.set_title("Brightness Histogram", color=text_color, fontsize=9, pad=5)

    # ── FFT magnitude (log scale) ──────────────────────────────────────────
    ax_fft.imshow(np.log1p(magnitude), cmap="inferno", origin="lower")
    ax_fft.axis("off")
    ax_fft.set_title("Fourier Magnitude (log scale)", color=text_color, fontsize=9, pad=5)

    # ── Frequency plots ────────────────────────────────────────────────────
    if show_radial:
        _plot_radial_spectrum(ax_rad_main, ax_rad_zoom, analytics, text_color, accent1)
    if show_freqhist:
        _plot_freq_histogram(ax_freqhist, analytics, text_color, accent2)

    plt.show()


print('✅ Analytics functions defined.')

### 4.9 Export

In [ ]:
def export_image(
    img: np.ndarray,
    filename: str,
    output_folder: str,
    fmt: str = "PNG",
    jpeg_quality: int = 95,
) -> str:
    """
    Save *img* to *output_folder*.
    The output filename is the original stem with '_processed' appended.
    Returns the full output path.
    """
    os.makedirs(output_folder, exist_ok=True)
    ext_map  = {"JPEG": ".jpg", "PNG": ".png", "TIFF": ".tiff",
                "BMP": ".bmp", "WEBP": ".webp", "GIF": ".gif"}
    ext      = ext_map.get(fmt.upper(), f".{fmt.lower()}")
    out_path = Path(output_folder) / f"{Path(filename).stem}_processed{ext}"
    save_kw  = {"quality": jpeg_quality} if fmt.upper() == "JPEG" else {}
    Image.fromarray(img).save(str(out_path), format=fmt, **save_kw)
    print(f"  💾 Saved: {out_path}")
    return str(out_path)


print('✅ export_image() defined.')

## 5. Reference Image (optional)

If `USE_REFERENCE_IMAGE = True`, this cell will analyse the reference image and use this data as the target settings for luminance (mean, peak, and floor luminance) and contrast (RMS).  

This overreds any manual values in Section 2.

This process is skipped if `USE_REFERENCE_IMAGE = False` and only prints a note that the reference image is not being used.

In [ ]:
if USE_REFERENCE_IMAGE:
    print(f"Loading reference image: {REFERENCE_IMAGE_PATH}")
    ref_arr = np.array(Image.open(REFERENCE_IMAGE_PATH).convert("RGB"), dtype=np.uint8)
    ref_analytics = compute_analytics(ref_arr, name="REFERENCE", show_plots=ENABLE_ANALYTICS)

    TARGET_MEAN_LUMINANCE  = ref_analytics["mean_luminance"]
    TARGET_PEAK_LUMINANCE  = ref_analytics["peak_luminance"]
    TARGET_FLOOR_LUMINANCE = ref_analytics["floor_luminance"]
    TARGET_RMS_CONTRAST    = ref_analytics["rms_contrast"]

    print(f"\nTargets derived from reference:")
    print(f"  Mean luminance  : {TARGET_MEAN_LUMINANCE:.2f}")
    print(f"  Peak luminance  : {TARGET_PEAK_LUMINANCE:.2f}")
    print(f"  Floor luminance : {TARGET_FLOOR_LUMINANCE:.2f}")
    print(f"  RMS contrast    : {TARGET_RMS_CONTRAST:.2f}")
else:
    print("Reference image not used — manual target values will be applied.")

## 6. Main Pipeline

Processes every image in order.  Each step only runs if its `ENABLE_*` flag is `True`.

**Step order and rationale:**
1. Greyscale — convert colour to luminance before any adjustments.
2. Face alignment — scale and centre based on landmark geometry.
3. Scale / crop — bring to target output dimensions.
4. Adjustments — luminance, contrast, frequency filtering on the full image.
5. Oval mask — composite adjusted image against background.  Must come last
   so the background colour is exact and unaffected by adjustments.

In [ ]:
print("═" * 58)
print(" STEP 1 — Loading images")
print("═" * 58)
images = load_images(INPUT_PATH, SUPPORTED_EXTENSIONS)

processed_images: List[Tuple[str, np.ndarray]] = []
analytics_results: Dict[str, Dict]             = {}
landmark_store:    Dict[str, Optional[np.ndarray]] = {}

for filename, img in images:

    print(f"\n{'─' * 58}")
    print(f" Processing: {filename}")
    print(f"{'─' * 58}")

    current   = img.copy()
    landmarks = None

    # ── Step 2: Greyscale ──────────────────────────────────────────────────
    if ENABLE_GREYSCALE:
        print("\n[2] Converting to greyscale…")
        current = to_greyscale(current)
        print("    ✓ Done")

    # ── Step 3: Face alignment ─────────────────────────────────────────────
    # Detects facial landmarks, then scales the image so the chosen reference
    # measure (e.g. eye width) matches FACE_SCALE_TARGET pixels, and translates
    # so the chosen centering reference (e.g. eye midpoint) lands at the
    # image centre plus any specified offset.
    if ENABLE_FACE_ALIGN:
        print("\n[3] Aligning face…")
        current, landmarks = align_face(
            current,
            center_mode  = FACE_CENTER_MODE,
            scale_mode   = FACE_SCALE_MODE,
            scale_target = FACE_SCALE_TARGET,
            offset_x     = FACE_OFFSET_X,
            offset_y     = FACE_OFFSET_Y,
        )
        if landmarks is not None:
            print(f"    ✓ Aligned  ({current.shape[1]}×{current.shape[0]} px after scaling)")

    # ── Step 4: Scale & crop ───────────────────────────────────────────────
    # Landmark coordinates are updated after each transform so they remain
    # accurate for the final image dimensions used in the analytics preview.
    if ENABLE_SCALING:
        h_pre, w_pre = current.shape[:2]
        current = scale_image(current, MAX_WIDTH, MAX_HEIGHT)
        print(f"\n[4a] Scaled to: {current.shape[1]}×{current.shape[0]} px")
        if landmarks is not None:
            scale = current.shape[1] / w_pre
            landmarks = landmarks * scale

    if ENABLE_CROPPING:
        h_pre, w_pre = current.shape[:2]
        crop_w_eff = min(DESIRED_W, w_pre)
        crop_h_eff = min(DESIRED_H, h_pre)
        start_x = (w_pre - crop_w_eff) // 2 if OFFSET_X is None else max(0, min(OFFSET_X, w_pre - crop_w_eff))
        start_y = (h_pre - crop_h_eff) // 2 if OFFSET_Y is None else max(0, min(OFFSET_Y, h_pre - crop_h_eff))
        current = crop_image(current, DESIRED_W, DESIRED_H, OFFSET_X, OFFSET_Y)
        print(f"\n[4b] Cropped to: {current.shape[1]}×{current.shape[0]} px")
        if landmarks is not None:
            landmarks = landmarks - np.array([start_x, start_y], dtype=np.float32)

    landmark_store[filename] = landmarks

    
    # ── Step 5: Adjustments ────────────────────────────────────────────────
    if ENABLE_ADJUSTMENTS:
        print("\n[5] Applying adjustments…")

        # Build the mask once so statistics are computed over the oval region only.
        # This ensures the target mean/contrast values reflect what is actually
        # visible in the final image, not the full rectangular frame.
        roi_mask = None
        if ENABLE_OVAL_MASK:
            H, W  = current.shape[:2]
            roi_mask = build_oval_mask(
                H, W,
                oval_w         = OVAL_WIDTH,
                oval_h         = OVAL_HEIGHT,
                offset_x       = OVAL_OFFSET_X,
                offset_y       = OVAL_OFFSET_Y,
                gradient_width = OVAL_GRADIENT_WIDTH,
                gradient_curve = OVAL_GRADIENT_CURVE,
                power          = OVAL_GRADIENT_POWER,
            )

        if TARGET_MEAN_LUMINANCE is not None:
            print(f"    • Mean luminance  → {TARGET_MEAN_LUMINANCE}")
            current = adjust_mean_luminance(current, TARGET_MEAN_LUMINANCE, mask=roi_mask)

        new_mean = float(
            np.average(compute_luminance(current), weights=roi_mask)
            if roi_mask is not None
            else compute_luminance(current).mean()
        )

        if TARGET_PEAK_LUMINANCE is not None:
            print(f"    • Peak luminance  → {TARGET_PEAK_LUMINANCE}")
            current = adjust_peak_luminance(current, TARGET_PEAK_LUMINANCE,
                                            mean_lum=new_mean, mask=roi_mask)

        if TARGET_FLOOR_LUMINANCE is not None:
            print(f"    • Floor luminance → {TARGET_FLOOR_LUMINANCE}")
            current = adjust_floor_luminance(current, TARGET_FLOOR_LUMINANCE,
                                             mean_lum=new_mean, mask=roi_mask)

        if TARGET_RMS_CONTRAST is not None:
            print(f"    • RMS contrast    → {TARGET_RMS_CONTRAST}")
            current = adjust_rms_contrast(current, TARGET_RMS_CONTRAST, mask=roi_mask)

        if ENABLE_FREQ_FILTER:
            print(f"    • Frequency filter: {FREQ_FILTER_TYPE}-pass  "
                  f"cutoff={FREQ_FILTER_CUTOFF}  rolloff={FREQ_FILTER_ROLLOFF}")
            current = apply_frequency_filter(
                current,
                filter_type = FREQ_FILTER_TYPE,
                cutoff      = FREQ_FILTER_CUTOFF,
                rolloff     = FREQ_FILTER_ROLLOFF,
            )

        print("    ✓ Done")
    
    
 
    # ── Step 6: Oval mask ──────────────────────────────────────────────────
    # Applied after all adjustments so the background colour is never
    # modified by upstream processing.
    if ENABLE_OVAL_MASK:
        print("\n[6] Applying oval mask…")
        current = apply_oval_mask(
            current,
            oval_w         = OVAL_WIDTH,
            oval_h         = OVAL_HEIGHT,
            offset_x       = OVAL_OFFSET_X,
            offset_y       = OVAL_OFFSET_Y,
            bg_color       = OVAL_BG_COLOR,
            gradient_width = OVAL_GRADIENT_WIDTH,
            gradient_curve = OVAL_GRADIENT_CURVE,
            power          = OVAL_GRADIENT_POWER,
        )
        print("    ✓ Done")

    # ── Step 7: Analytics ─────────────────────────────────────────────────
    if ENABLE_ANALYTICS:
        print("\n[7] Computing analytics…")
        analytics = compute_analytics(current, name=filename, show_plots=False)
        _plot_analytics(
            current, analytics, name=filename,
            landmarks=landmarks if SHOW_LANDMARKS else None,
        )
        analytics_results[filename] = analytics

    processed_images.append((filename, current))

    # ── Step 8: Export ────────────────────────────────────────────────────
    if ENABLE_EXPORT:
        print("\n[8] Exporting…")
        export_image(current, filename, OUTPUT_FOLDER,
                     fmt=OUTPUT_FORMAT, jpeg_quality=JPEG_QUALITY)

print(f"\n{'═' * 58}")
print(f" ✅ Pipeline complete — {len(processed_images)} image(s) processed.")
print(f"{'═' * 58}")

## 7. Visual Comparison

Side-by-side view of every original and processed image.

In [ ]:
n = len(images)
fig, axes = plt.subplots(n, 2, figsize=(12, 5 * n), facecolor="#1a1a2e")
if n == 1:
    axes = [axes]

for i, ((filename, orig), (_, proc)) in enumerate(zip(images, processed_images)):
    ax_orig, ax_proc = axes[i]
    for ax in (ax_orig, ax_proc):
        ax.set_facecolor("#16213e")
        ax.axis("off")
    ax_orig.imshow(orig)
    ax_orig.set_title(f"Original — {filename}", color="white", fontsize=10, pad=6)
    ax_proc.imshow(proc)
    ax_proc.set_title(f"Processed — {filename}", color="#00b4d8", fontsize=10, pad=6)

plt.tight_layout()
plt.show()

## 8. Analytics Summary Table

Key metrics for every processed image in a single printout, suitable for
copying into a spreadsheet or lab notebook.

In [ ]:
if analytics_results:
    # Check whether band power data is available
    has_bands = any(r.get("band_power") for r in analytics_results.values())

    if has_bands:
        print(f"{'Image':<35} {'Mean':>7} {'Peak':>7} {'Floor':>7} {'RMS':>7} "
              f"{'<1cpd':>8} {'1-6cpd':>8} {'6-12cpd':>9} {'>12cpd':>8}")
        print("─" * 105)
        for fname, a in analytics_results.items():
            bp = a.get("band_power", {})
            print(
                f"{fname:<35} "
                f"{a['mean_luminance']:7.2f} "
                f"{a['peak_luminance']:7.2f} "
                f"{a['floor_luminance']:7.2f} "
                f"{a['rms_contrast']:7.2f} "
                f"{bp.get('below_1cpd',  {}).get('percent', 0):7.1f}% "
                f"{bp.get('1_to_6cpd',   {}).get('percent', 0):7.1f}% "
                f"{bp.get('6_to_12cpd',  {}).get('percent', 0):7.1f}%  "
                f"{bp.get('above_12cpd', {}).get('percent', 0):7.1f}%"
            )
    else:
        print(f"{'Image':<35} {'Mean':>7} {'Peak':>7} {'Floor':>7} {'RMS':>7}")
        print("─" * 65)
        for fname, a in analytics_results.items():
            print(
                f"{fname:<35} "
                f"{a['mean_luminance']:7.2f} "
                f"{a['peak_luminance']:7.2f} "
                f"{a['floor_luminance']:7.2f} "
                f"{a['rms_contrast']:7.2f}"
            )
else:
    print("Analytics were not enabled (ENABLE_ANALYTICS = False).")